# Determine threshold for HBN windows

In [ ]:
import pandas as pd 


In [17]:
# log_file = "/data/linsk/temp_prep/logs/quality_metrics.csv"
log_file = "/data/linsk/hbn_preprocessed/logs/quality_metrics.csv"

In [18]:
# load txt to pandas, header first line
df = pd.read_csv(log_file, sep=",", header=0)

In [19]:
df.head(2)

,filename,window_start_time,window_end_time,oha1,thv1,chv1,bcr1,oha2,thv2,chv2,bcr2
0,/data/agjma/HBN_EEG_Raw/ds005516/sub-NDARWT989...,0.0,30.0,0.159538,0.294667,0.057143,0.114286,0.161641,0.271800,0.042105,0.031579
1,/data/agjma/HBN_EEG_Raw/ds005516/sub-NDARWT989...,15.0,45.0,0.228537,0.372533,0.123810,0.104762,0.230983,0.294067,0.107527,0.010753


**Current thresholds:**
- oha_threshold=40e-6, 
- thv_threshold=40e-6, 
- chv_threshold=80e-6,
- min_unique_ratio=0.001,

- <span style="color:red">(oha < 0.8)</span> Overall High Amplitude - frac samples across channels that exceed a high amplitude threshold (oha_threshold). Large values - spikes or artifacts.
- <span style="color:red">(thv < 0.5)</span> Temporal High Variance - frac time points where standard deviation across channels exceeds thv_threshold.
- <span style="color:red">(chv < 0.5)</span> Channel High Variance - frac chans where standard deviation exceeds chv_threshold.
- <span style="color:red">(bcr < 0.8)</span> Bad Channel Ratio

*True* is good quality; if all four metrics are below threshold, return True; if at least one is above threshold, return False

In [20]:
# get stats: 
df.describe()

,window_start_time,window_end_time,oha1,thv1,chv1,bcr1,oha2,thv2,chv2,bcr2
count,206622.000000,206622.000000,206622.000000,206622.000000,206622.000000,206622.000000,143834.000000,143834.000000,143834.000000,143834.000000
mean,211.616866,241.616866,0.130255,0.303206,0.134406,0.140521,0.064098,0.070113,0.024994,0.012951
std,319.524542,319.524542,0.175432,0.335528,0.268389,0.136101,0.066556,0.095677,0.058815,0.017762
min,0.000000,30.000000,0.000000,0.000000,0.000000,0.019048,0.000000,0.000000,0.000000,0.000000
25%,60.000000,90.000000,0.033026,0.026133,0.000000,0.085714,0.017110,0.000867,0.000000,0.000000
50%,135.000000,165.000000,0.071778,0.150100,0.019048,0.123810,0.042280,0.024733,0.000000,0.010000
75%,240.000000,270.000000,0.148285,0.521333,0.095238,0.161905,0.088275,0.106200,0.020408,0.020408
max,3540.000000,3570.000000,0.984358,1.000000,0.990476,1.800000,0.658696,0.521800,1.000000,0.400000


## how many dropped: 


In [21]:
df["dropped1"] = ~((df["oha1"] < 0.8) & (df["thv1"] < 0.5) & (df["chv1"] < 0.5) & (df["bcr1"] < 0.8))
df["dropped2"] = ~((df["oha2"] < 0.8) & (df["thv2"] < 0.5) & (df["chv2"] < 0.5) & (df["bcr2"] < 0.8))


In [22]:
df["dropped"] = df["dropped1"] | df["dropped2"]


In [23]:
df.head()

,filename,window_start_time,window_end_time,oha1,thv1,chv1,bcr1,oha2,thv2,chv2,bcr2,dropped1,dropped2,dropped
0,/data/agjma/HBN_EEG_Raw/ds005516/sub-NDARWT989...,0.0,30.0,0.159538,0.294667,0.057143,0.114286,0.161641,0.271800,0.042105,0.031579,False,False,False
1,/data/agjma/HBN_EEG_Raw/ds005516/sub-NDARWT989...,15.0,45.0,0.228537,0.372533,0.123810,0.104762,0.230983,0.294067,0.107527,0.010753,False,False,False
2,/data/agjma/HBN_EEG_Raw/ds005516/sub-NDARWT989...,30.0,60.0,0.290682,0.320200,0.057143,0.066667,0.288975,0.230467,0.031915,0.010638,False,False,False
3,/data/agjma/HBN_EEG_Raw/ds005516/sub-NDARWT989...,45.0,75.0,0.299902,0.374400,0.047619,0.076190,0.292301,0.215600,0.010870,0.032609,False,False,False
4,/data/agjma/HBN_EEG_Raw/ds005516/sub-NDARWT989...,60.0,90.0,0.127196,0.190067,0.019048,0.095238,0.128406,0.137733,0.010101,0.020202,False,False,False


In [24]:
total = len(df)
dropped_count = df["dropped"].sum()
kept_count = total - dropped_count

dropped_pct = dropped_count / total * 100
kept_pct = kept_count / total * 100

print(f"Dropped: {dropped_count} ({dropped_pct:.1f}%)")
print(f"Kept: {kept_count} ({kept_pct:.1f}%)")


Dropped: 62891 (30.4%)
Kept: 143731 (69.6%)


In [25]:
thresholds = {"oha": 0.8, "thv": 0.5, "chv": 0.5, "bcr": 0.8}

for i in [1, 2]:
    print(f"\nStage {i}:")
    for metric, thr in thresholds.items():
        col = f"{metric}{i}"
        pct_exceed = (df[col] >= thr).mean() * 100
        print(f"  {col}: {pct_exceed:.1f}% exceed threshold")



Stage 1:
  oha1: 2.3% exceed threshold
  thv1: 25.9% exceed threshold
  chv1: 10.0% exceed threshold
  bcr1: 0.6% exceed threshold

Stage 2:
  oha2: 0.0% exceed threshold
  thv2: 0.0% exceed threshold
  chv2: 0.0% exceed threshold
  bcr2: 0.0% exceed threshold


In [ ]:
# 30 percent are dropped
# most bc of thv
# fraction of time points in the window where std between channels is higher than 40e-6. ie channles disagree too strongly in above 50% of the cases. 
